# Horizon-shift signal vs the foreground subspace

Antenna-position error changes the antenna temperature by
$\Delta T_\mathrm{ant}(\nu)$. This figure asks how much of that change lies
*outside* the subspace the foregrounds occupy: the $\Delta T_\mathrm{ant}$
spectra (top row, +1 m East/North/Up at 24 LSTs) are decomposed on the
foreground spectral modes -- the same SVD modes as `foreground_svd.npz`,
carried here as `Vh` -- and the bottom row plots the RMS of the component
orthogonal to the leading $N$ of them, for each simulated magnitude.

Grey is the 21 cm ensemble in the same orthogonal complement (5-95% of the
models, median dashed). It is a *yardstick* for how small a millikelvin number
has to be, not a signal being recovered here.

**Three things the bottom row is for.**

*Amplitude is not the story.* The $\Delta T$ are large -- several K at the
bottom of the band -- and ~99.8% of that power sits in the two leading
eigenmodes of the unperturbed antenna temperature. That statistic flatters the
result: the foregrounds leak ~1e-5 of their power past those two modes and a
displacement leaks ~1e-3, so a position error shares the foregrounds' leading
modes but has a spectrum roughly half as steep in log, and does not compress
the way they do. `print_compressibility` prints both sides of that.

*The floor is a property of a horizon that is known.* The modes are the right
singular vectors of the nominal antenna temperature, so the foregrounds are
scored in their own optimal basis while a displacement is not. That asymmetry
is not sampling -- a basis built from half the sidereal day describes the other
half with no penalty -- it is specific to the horizon. Decompose each displaced
sky in *its own* basis and the floor barely moves (`print_floor_table`): a
displaced antenna temperature is not a more complex object, and would be
described just as economically if the displacement were known. What a position
error costs is the mismatch, and that is the number to quote -- an unmodelled
+1 m vertical error roughly doubles the floor.

*The vertical axis sets the requirement.* Only the vertical response is
proportional to displacement, because a vertical shift lowers the horizon by a
near-uniform offset; East and North are set by where the cliff edges fall in
azimuth and are neither linear nor symmetric. Up is the binding axis at every
$N$, so the extrapolation is sound and the requirement is a vertical one.

**What this is not.** This figure is not a proposed
analysis; it decomposes a *simulated* nominal instrument. Mode counts appear
below only because the residual has to be read somewhere, and they are
fragile: 99 per cent of what the vertical displacement leaves after
$N_\mathrm{HAND}$ modes is a single eigenmode, so filtering one mode more
drops it by a factor of ten and every count either side of that mode
disagrees. A residual left by a projection
of fixed depth is not evidence of a cosmological signal: excess with respect to
a foreground model is only as trustworthy as the instrument model behind it.
EIGSEP's analysis marginalises over antenna position inside a differentiable
forward model instead, and the sensitivities here are what set its priors.

Crossings below use a *stays below* rule: the smallest $N$ at which the
worst-LST residual is under the median retained signal and remains so for every
larger $N$. It is the conservative choice, and it states something about every
larger $N$ rather than about one; a first-crossing rule can report a transient
that the curve later climbs back out of.

Grey is the 21 cm ensemble **in antenna temperature**: the models are
multiplied by the beam-weighted open-sky fraction $\eta = 1 - f_\mathrm{gnd}$
(0.36-0.55 across the band) before filtering, because $\Delta T_\mathrm{ant}$
and the modes are both uncorrected antenna temperature and an isotropic signal
reaches that observable attenuated. It is not the sky-referred global signal,
and it is 2.2 times smaller than one.

Data: `horizon_shift.npz`. Full derivation from the raw simulation output:
`mock_analysis/horizon_position/notebooks/horizon_shift.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

In [ ]:
d = np.load("horizon_shift.npz", allow_pickle=True)
freqs = d["freqs_MHz"]
lst = d["lst_hr"]                 # LST [h] of each plotted spectrum
t21 = d["t21_pct"]                # (3, n_modes) retained 21 cm RMS [K], 5/50/95
dT_disp = d["dT_disp"]            # (3 axis, 3 mag, n_lst, n_freq) dT_ant [K]
mags = d["mags_m"]                # displacement magnitudes [m]
top_mag = float(d["top_mag_m"])   # the magnitude the spectra row draws
Vh = d["Vh"]                      # (n_freq, n_freq) foreground spectral modes
labels = [str(x) for x in d["labels"]]
N_ANCHOR = int(d["n_anchor"])     # the anchor depth, N_ANCHOR
n_hand = int(d["n_hand"])         # where the position error overtakes the
                                  # foreground floor, one mode inside it
s_fg = d["s_fg"]                  # singular values of the nominal waterfall
pos_names = [str(x) for x in d["pos_names"]]
floor_own = d["floor_own"]        # floor at n_hand, each sky in its OWN basis
floor_nom = d["floor_nom"]        # ... and the same skies in the nominal basis
cv_penalty = float(d["cv_penalty"])
n_time = int(d["n_time"])
n_modes = np.arange(19)           # foreground modes filtered (x-axis)
print(dT_disp.shape, "at magnitudes", mags, "m; spectra row =", top_mag, "m")
print("LSTs", np.round(lst, 1))

In [ ]:
def resid_curves(dT_axis, Vh, n_modes):
    """Per-row residual RMS over frequency [K] after filtering the leading N modes.

    ``dT_axis`` is (n_row, n_freq); rows are LSTs here and models for the 21 cm
    ensemble. Worst-case (per-row), not pooled -- the position systematic is
    quoted against its worst LST.
    """
    n_f = dT_axis.shape[1]
    coeff = dT_axis @ Vh.T
    return np.array([np.sqrt(np.sum(coeff[:, N:] ** 2, axis=1) / n_f)
                     for N in n_modes])                  # (n_modes, n_row)


def stays_below(curve, ref, n_modes):
    """Smallest N with curve < ref there and at every larger N on the axis.

    Not first-crossing: both fall with N and cross more than once, so a
    first-crossing rule reports an N the curve later climbs back above.
    """
    below = curve < ref
    return next((N for N in n_modes if below[N:].all()), None)


def make_figure(freqs, lst, t21, dT_disp, mags, top_mag, Vh, labels, n_modes, out_path):
    """The 2x3 horizon-shift figure. Returns the figure; also writes ``out_path``."""
    CMAP, norm = "twilight", Normalize(0, 24)
    C_21 = "0.40"                     # 21 cm band: grey and dashed, so it reads as
                                      # a benchmark in both rows without competing
                                      # with either row's colour encoding.
    D_COL = ["#6baed6", "#2171b5", "#08306b"]   # sequential: displacement is ordered
    cmap = plt.get_cmap(CMAP)
    n_f = freqs.size
    i_top = int(np.argmin(np.abs(mags - top_mag)))
    dT = dT_disp[:, i_top]                                  # (3, n_lst, n_freq)

    fig, axes = plt.subplots(
        2, 3, figsize=(7.3, 3.6),
        gridspec_kw=dict(height_ratios=[1.6, 1.15]),
        layout="constrained",
    )
    for col, lab in enumerate(labels):
        at, ab = axes[0, col], axes[1, col]
        for j in range(lst.size):                           # top: dT(nu) spectra
            at.plot(freqs, dT[col, j], color=cmap(norm(lst[j])), lw=0.7, alpha=0.9)
        at.axhline(0, color="0.5", lw=0.6, ls="--", zorder=0)
        at.set_title(lab, fontsize=8.5)
        at.set_xlabel("Frequency [MHz]", fontsize=8)
        at.grid(alpha=0.2); at.tick_params(labelsize=7)
        # Headroom for the magnitude tag. The three panels span very different
        # ranges and the curves reach the top of a different one in each, so the
        # tag needs space made for it rather than a corner that happens to be free.
        lo, hi = dT[col].min(), dT[col].max()
        at.set_ylim(lo - 0.05 * (hi - lo), hi + 0.30 * (hi - lo))
        at.text(0.97, 0.94, f"$+{top_mag:g}$ m", transform=at.transAxes,
                fontsize=6.5, color="0.35", ha="right", va="top")

        for k in range(mags.size):                          # bottom: all magnitudes
            rc = resid_curves(dT_disp[col, k], Vh, n_modes)  # (n_modes, n_lst)
            for j in range(lst.size):
                ab.plot(n_modes, rc[:, j], color=D_COL[k], lw=0.55, alpha=0.6)
        ab.fill_between(n_modes, t21[0], t21[2], color=C_21, alpha=0.25, lw=0, zorder=0)
        ab.plot(n_modes, t21[1], color=C_21, lw=1.4, ls="--", zorder=1)
        ab.set_yscale("log")
        ab.set_xlabel("Foreground modes filtered", fontsize=8)
        ab.grid(True, which="both", ls=":", lw=0.5, alpha=0.55)
        ab.set_xlim(0, n_modes[-1]); ab.set_ylim(3e-5, 30); ab.tick_params(labelsize=7)

    axes[0, 0].set_ylabel(r"$\Delta T_\mathrm{ant}$ [K]", fontsize=8)
    axes[1, 0].set_ylabel("Residual RMS [K]", fontsize=8)
    handles = [Line2D([], [], color=D_COL[k], lw=2, label=f"{mg:g} m")
               for k, mg in enumerate(mags)]
    handles.append(Line2D([], [], color=C_21, lw=1.4, ls="--", label="21-cm models"))
    # Upper right, not lower left: every curve descends with N, so the top-right
    # corner is the one reliably empty region in all three residual panels, while
    # the lower left still carries the 0.1 m tails.
    axes[1, 0].legend(handles=handles, fontsize=5.8, loc="upper right", ncol=2,
                      framealpha=0.9, handlelength=1.4, columnspacing=0.9,
                      borderpad=0.3, labelspacing=0.25)
    for col in (1, 2):
        axes[1, col].tick_params(labelleft=False)

    sm = ScalarMappable(norm=norm, cmap=CMAP)
    cb = fig.colorbar(sm, ax=axes[0, :], pad=0.012, fraction=0.03)
    cb.set_label("LST [h]", fontsize=8); cb.set_ticks(np.arange(0, 25, 6))
    cb.ax.tick_params(labelsize=7)

    fig.savefig(out_path, bbox_inches="tight", dpi=600)
    return fig


def print_summary(dT_disp, mags, top_mag, Vh, labels, lst, t21, n_modes,
                  n_anchor, n_hand):
    """Every number the paper text quotes from this figure.

    ``n_anchor`` is where the foreground floor clears the median retained
    signal; ``n_hand`` is one mode inside it, where the position error
    overtakes the foreground floor. The anatomy of what escapes is taken at
    ``n_hand`` -- past it the residual is the position error, which is what
    makes "what survives the filter" the right question there. At ``n_anchor``
    the spike below has already been filtered out.
    """
    n_f = dT_disp.shape[-1]
    i_top = int(np.argmin(np.abs(mags - top_mag)))
    worst = np.array([[resid_curves(dT_disp[c, k], Vh, n_modes).max(axis=1)
                       for k in range(mags.size)]
                      for c in range(len(labels))])        # (axis, mag, n_modes)

    med21 = t21[1, n_anchor] * 1e3
    print(f"worst-LST residual, median retained 21 cm at N={n_anchor} is {med21:.2f} mK\n")
    print(f"{'axis':7s}{'shift':>8s}{'unfiltered':>12s}{'at N=%d' % n_anchor:>10s}"
          f"{'stays below from':>18s}")
    for c, lab in enumerate(labels):
        for k, mg in enumerate(mags):
            w = worst[c, k]
            print(f"{lab if k == 0 else '':7s}{mg:7g}m{w[0]*1e3:11.1f} mK"
                  f"{w[n_anchor]*1e3:9.2f} mK"
                  f"{stays_below(w, t21[1], n_modes):15d} modes")

    worst_all = worst[:, i_top].max(axis=0)
    n_sys = stays_below(worst_all, t21[1], n_modes)
    print(f"\nFig. 1 sets N = {n_anchor} on the foreground residual alone. Folding in "
          f"the +{top_mag:g} m position systematic costs {n_sys - n_anchor} further "
          f"mode(s): worst axis/LST {worst_all[n_anchor]*1e3:.2f} mK at N = {n_anchor} "
          f"(median retained {med21:.2f} mK), {worst_all[n_sys]*1e3:.2f} mK at "
          f"N = {n_sys} (median retained {t21[1, n_sys]*1e3:.2f} mK).")

    # Does the residual scale with displacement? Only on the vertical axis.
    print()
    for c, lab in enumerate(labels):
        r = worst[c, :, n_anchor]
        dev = np.abs((r[1:] / r[:-1]) / (mags[1:] / mags[:-1]) - 1) * 100
        print(f"{lab:7s} deviation from proportionality per decade: "
              + ", ".join(f"{x:.1f}%" for x in dev))

    i_up = labels.index("Up")
    med_hand = t21[1, n_hand] * 1e3
    spec = 0.1 * med_hand / (worst[i_up, i_top, n_hand] * 1e3) * top_mag
    print(f"\nUp is linear, and is the binding axis at every N. Holding its injection "
          f"to a tenth of the median retained signal at N = {n_hand} needs the "
          f"vertical position known to {spec:.2f} m.")

    # The two halves of the message, as numbers, taken at the handover.
    cu = dT_disp[2, i_top] @ Vh.T                          # Up, every LST
    j = int(np.argmax(np.sqrt(np.sum(cu[:, n_hand:]**2, axis=1))))
    mode_mK = np.abs(cu[j]) / np.sqrt(n_f) * 1e3
    lead = np.sum(cu[j, :2]**2) / np.sum(cu[j]**2)
    spike = int(np.argmax(mode_mK[n_hand:])) + n_hand
    tail = np.sum(mode_mK[n_hand:]**2)
    print(f"\nUp +{top_mag:g} m at LST {lst[j]:.0f} h: {lead*100:.1f}% of its power "
          f"sits in the two leading foreground modes -- mostly just more foreground. But "
          f"after filtering {n_hand} modes, {mode_mK[spike]**2/tail*100:.0f}% of what "
          f"remains is mode {spike+1} alone, at {mode_mK[spike]:.2f} mK against a "
          f"{med_hand:.2f} mK median retained signal, {mode_mK[spike]/med_hand:.1f}x "
          f"its magnitude. Filtering that one mode is what "
          f"drops the vertical under the median, so a fixed-depth filter set either "
          f"side of it reports a different result.")


def print_floor_table(pos_names, floor_own, floor_nom, cv_pen, n_anchor,
                      n_time, n_f):
    """How much of the floor is the foregrounds, and how much is the basis?"""
    base = floor_own[0]
    print(f"nominal floor at N={n_anchor}: {base * 1e3:.3f} mK")
    print(f"held-out LST penalty: {cv_pen:.2f}x   ({n_time} spectra x {n_f} "
          f"channels; {n_anchor} of a possible {min(n_time, n_f)} modes)")
    print(f"\n{'displacement':14s}{'own basis':>13s}{'nominal basis':>16s}"
          f"{'floor':>9s}")
    for nm, o, nb in zip(pos_names, floor_own, floor_nom):
        if nm == pos_names[0]:
            continue
        print(f"{nm:14s}{o * 1e3:10.3f} mK{nb * 1e3:13.3f} mK{nb / base:8.2f}x")


def print_compressibility(s_fg, dT_disp, Vh, labels, mags, top_mag,
                          n_time, n_f, n_anchor):
    """Is a position error 'more foreground'? Only in the crudest sense."""
    i_top = int(np.argmin(np.abs(mags - top_mag)))
    fg = s_fg ** 2
    hdr = (f"{'':12s}{'leak past 2':>13s}{'decades 1-10':>14s}{'RMS':>11s}"
           f"{'floor':>11s}{'compression':>13s}")
    print(hdr)
    rms = np.sqrt(fg.sum() / (n_time * n_f))
    flr = np.sqrt(fg[n_anchor:].sum() / (n_time * n_f))
    print(f"{'foregrounds':12s}{1 - fg[:2].sum() / fg.sum():13.2e}"
          f"{np.log10(fg[0] / fg[9]):14.1f}{rms:9.1f} K{flr * 1e3:8.2f} mK"
          f"{rms / flr:13.0f}")
    for c, lab in enumerate(labels):
        co = dT_disp[c, i_top] @ Vh.T
        p = (co ** 2).sum(axis=0)
        n_row = co.shape[0]
        rms = np.sqrt(p.sum() / (n_row * n_f))
        flr = np.sqrt(p[n_anchor:].sum() / (n_row * n_f))
        print(f"{lab + ' ' + '%g' % top_mag + ' m':12s}"
              f"{1 - p[:2].sum() / p.sum():13.2e}"
              f"{np.log10(p[0] / p[9]):14.1f}{rms:9.3f} K{flr * 1e3:8.2f} mK"
              f"{rms / flr:13.0f}")


def print_handover(fg_resid, dT_disp, Vh, mags, top_mag, labels, n_modes, n_f):
    """Where does the term limiting the residual change hands?

    The foreground floor is an in-sample optimum, so it falls faster with N
    than anything not used to build the basis. Past the point where an
    unmodelled position error overtakes it, filtering deeper removes signal
    from a residual that is no longer made of foreground -- which is the answer
    to "why not just remove every mode on the plot".
    """
    k = int(np.argmin(np.abs(mags - top_mag)))
    up = dT_disp[labels.index("Up"), k] @ Vh.T
    sys_r = np.array([np.sqrt(np.mean(np.sum(up[:, N:] ** 2, axis=1) / n_f))
                      for N in n_modes])
    first = next(N for N in n_modes if sys_r[N] >= fg_resid[N])
    stays = next(N for N in n_modes if (sys_r[N:] > fg_resid[N:]).all())
    print(f"{'N':>3}{'fg floor':>12}{'+%g m up' % top_mag:>12}{'limited by':>13}")
    for N in n_modes:
        who = "foregrounds" if fg_resid[N] > sys_r[N] else "position"
        print(f"{N:>3}{fg_resid[N] * 1e3:11.4f}{sys_r[N] * 1e3:11.4f}  {who:>12}")
    print(f"\nforegrounds larger by {fg_resid[first - 1] / sys_r[first - 1]:.1f}x "
          f"at N={first - 1}; the two cross at N={first}; from N={stays} the "
          f"position error is consistently larger, reaching "
          f"{sys_r[-1] / fg_resid[-1]:.1f}x at N={n_modes[-1]}")
    return first, stays

In [ ]:
fig = make_figure(freqs, lst, t21, dT_disp, mags, top_mag, Vh,
                  labels, n_modes, "horizon_shift.pdf")
print_summary(dT_disp, mags, top_mag, Vh, labels, lst, t21, n_modes, N_ANCHOR,
              n_hand)
print()
print_floor_table(pos_names, floor_own, floor_nom, cv_penalty, N_ANCHOR,
                  n_time, freqs.size)
print()
print_compressibility(s_fg, dT_disp, Vh, labels, mags, top_mag,
                      n_time, freqs.size, N_ANCHOR)
print()
fg_resid = np.array([np.sqrt(np.sum(s_fg[N:] ** 2) / (n_time * freqs.size))
                     for N in n_modes])
print_handover(fg_resid, dT_disp, Vh, mags, top_mag, labels, n_modes,
               freqs.size)